# Chapter 09 Demo: Grounding Agents in Evidence
## Magid's Newsroom RAG System — Triggering the Failure

**Design of Agentic Systems with Case Studies**  
*Aravind Balaji | INFO 7375: Prompt Engineering for Generative AI*

---

### What this notebook demonstrates

Two RAG pipelines process the **same** broadcast script with the **same** LLM.  
- **Pipeline A** (naive) has no context adherence measurement.  
- **Pipeline B** (scored) measures quote fidelity, attribution accuracy, and semantic fidelity.

The failure: Pipeline A produces fluent output that subtly misrepresents the source. Pipeline B detects and flags the deviations.

### The 0.681 gap
Token-overlap deviation = **0.871** (FLAGGED). Semantic deviation = **0.190** (NOT FLAGGED). Delta = **0.681**. This is why generic metrics miss domain-specific failures.

In [ ]:
# !pip install -r requirements.txt

In [ ]:
import os, json, re
import numpy as np

from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

os.environ["ANTHROPIC_API_KEY"] = "your-api-key-here"  # Replace before running

llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0.3)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Setup complete. Same LLM and embeddings for ALL pipelines.")

---
## Part 1 — The 0.681 gap (worked example)

In [ ]:
SOURCE_SENTENCE = (
    'Councilmember Reyes stopped short of endorsing the rezoning, '
    'saying the proposal needed "more community input" before she could support it.'
)
FABRICATED_OUTPUT = (
    '"I support bringing more housing to this neighborhood," said '
    'Councilmember Reyes, who backed the measure pending additional review.'
)

print("SOURCE:", SOURCE_SENTENCE)
print("\nFABRICATED:", FABRICATED_OUTPUT)
print("\nThe output reverses the valence. Reyes dissented; the fabrication says she supported.")

In [ ]:
# === Shared utility functions ===

def tokenize(text):
    return set(re.sub(r'[^\w\s]', '', text.lower()).split())

def jaccard(a, b):
    ta, tb = tokenize(a), tokenize(b)
    inter = ta & tb
    union = ta | tb
    return len(inter) / len(union) if union else 0.0

def cosine_sim(text_a, text_b):
    va = np.array(embeddings.embed_query(text_a))
    vb = np.array(embeddings.embed_query(text_b))
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))

def extract_quotes(text):
    return re.findall(r'"([^"]+)"', text) + re.findall(r'\u201c([^\u201d]+)\u201d', text)

print("Utility functions loaded.")

In [ ]:
# === Compute the gap ===

j = jaccard(SOURCE_SENTENCE, FABRICATED_OUTPUT)
token_dev = 1 - j

cos = cosine_sim(SOURCE_SENTENCE, FABRICATED_OUTPUT)
sem_dev = 1 - cos

delta = token_dev - sem_dev

src_tok = tokenize(SOURCE_SENTENCE)
out_tok = tokenize(FABRICATED_OUTPUT)
inter = src_tok & out_tok

print("TOKEN OVERLAP (Jaccard)")
print(f"  Source tokens ({len(src_tok)}): {sorted(src_tok)}")
print(f"  Output tokens ({len(out_tok)}): {sorted(out_tok)}")
print(f"  Intersection ({len(inter)}): {sorted(inter)}")
print(f"  Jaccard = {j:.3f}  |  Token deviation = {token_dev:.3f}")
print(f"  Threshold > 0.5: {'FLAGGED ✓' if token_dev > 0.5 else 'NOT FLAGGED ✗'}")

print(f"\nSEMANTIC SIMILARITY (embedding cosine)")
print(f"  Cosine = {cos:.3f}  |  Semantic deviation = {sem_dev:.3f}")
print(f"  Threshold > 0.5: {'FLAGGED ✓' if sem_dev > 0.5 else 'NOT FLAGGED ✗'}")

print(f"\n{'='*55}")
print(f"  Token deviation:    {token_dev:.3f}  →  FLAGGED")
print(f"  Semantic deviation: {sem_dev:.3f}  →  NOT FLAGGED")
print(f"  Delta:              {delta:.3f}")
print(f"{'='*55}")

---
## Part 2 — Broadcast script (test input)

In [ ]:
BROADCAST_SCRIPT = """CITY COUNCIL HOUSING VOTE — BROADCAST SCRIPT
Reporter: Maria Chen | Segment: 1:45 | Air Date: Tuesday

The Millbrook City Council voted six to three Tuesday night to approve a 
controversial rezoning plan that would allow high-density housing development 
along the Cedar Avenue corridor.

The plan, which has drawn sharp debate over the past four months, would permit 
buildings up to eight stories in a neighborhood currently zoned for single-family 
homes. According to city planning documents, the rezoning could add up to 
twelve hundred new residential units over the next decade.

Mayor Linda Park called the vote "a turning point for our city's housing 
crisis" and said the plan would address a shortage that has pushed average 
rents up twenty-three percent since 2021, according to the Regional Housing 
Authority.

Councilmember Reyes stopped short of endorsing the rezoning, saying the 
proposal needed "more community input" before she could support it. She was 
among the three dissenting votes.

However, neighborhood association president David Okafor warned that the 
development "will fundamentally change the character of Cedar Avenue" and 
urged the council to require a traffic impact study before construction begins.

City staff estimate the first permits could be issued within ninety days, 
pending environmental review."""

print("Script loaded. Key elements:")
print('  Q1: Park — "a turning point for our city\'s housing crisis"')
print('  Q2: Reyes — "more community input"')
print('  Q3: Okafor — "will fundamentally change the character of Cedar Avenue"')
print('  S1: 1,200 units → city planning documents')
print('  S2: 23% since 2021 → Regional Housing Authority')
print('  Qualifier: Reyes "stopped short of endorsing" — dissented')

---
## Part 3 — Pipeline A: naive RAG (no adherence measurement)

In [ ]:
def run_pipeline(script, model, collection_name="pipe"):
    """Core pipeline: chunk → retrieve → generate. No scoring."""
    splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
    chunks = splitter.create_documents([script])
    vs = Chroma.from_documents(chunks, embeddings, collection_name=collection_name)
    retrieved = vs.similarity_search("city council housing vote rezoning", k=5)
    context = "\n\n".join([c.page_content for c in retrieved])
    prompt = f"""You are a digital news editor. Rewrite the following broadcast 
script as a web news story. Make it engaging for digital readers. Include direct 
quotes where available. Write 3-4 paragraphs.

Source material:\n{context}\n\nWeb story:"""
    response = model.invoke([HumanMessage(content=prompt)])
    vs.delete_collection()
    return response.content

print("Running Pipeline A (naive, no measurement)...\n")
naive_output = run_pipeline(BROADCAST_SCRIPT, llm, "naive")

print("PIPELINE A OUTPUT")
print("="*60)
print(naive_output)
print("\nAdherence check: ABSENT. Deviations: UNKNOWN.")

---
## Part 4 — Pipeline B: scored RAG (three-axis adherence + token verification)

In [ ]:
def three_axis_scorer(source, generated):
    """LLM-based three-axis context adherence scorer."""
    prompt = f"""You are a journalistic accuracy auditor. Compare GENERATED against SOURCE.

SOURCE:\n{source}\n\nGENERATED:\n{generated}

Score 1-5 each:
QUOTE_FIDELITY: Are quotes word-for-word exact? Fabricated quote = 1.
ATTRIBUTION_ACCURACY: Facts attributed to correct sources?
SEMANTIC_FIDELITY: Meaning preserved? Qualifiers retained?

Check specifically:
- Park: "a turning point for our city's housing crisis"
- Reyes: "more community input" (she DISSENTED)
- Okafor: "will fundamentally change the character of Cedar Avenue"
- 1,200 units → city planning documents
- 23% → Regional Housing Authority

JSON only:
{{"quote_fidelity": int, "attribution_accuracy": int, "semantic_fidelity": int,
  "deviations": ["list each"], "decision": "PASS" or "FLAG",
  "fix_suggestions": ["list each"]}}

If any score < 3 → FLAG. Fabricated quote → automatic FLAG."""
    resp = llm.invoke([HumanMessage(content=prompt)])
    try:
        return json.loads(re.search(r'\{.*\}', resp.content, re.DOTALL).group())
    except:
        return {"quote_fidelity": 1, "attribution_accuracy": 1, "semantic_fidelity": 1,
                "deviations": ["Parse error"], "decision": "FLAG", "fix_suggestions": []}

def token_quote_check(source, output):
    """Deterministic token-level quote verification (no LLM)."""
    src_q = extract_quotes(source)
    out_q = extract_quotes(output)
    results = []
    for oq in out_q:
        found = oq.lower() in source.lower() or any(
            oq.lower() in sq.lower() or sq.lower() in oq.lower() for sq in src_q)
        best_j = max((jaccard(oq, sq) for sq in src_q), default=0)
        best_cos = max((cosine_sim(oq, sq) for sq in src_q), default=0) if src_q else 0
        results.append({
            "quote": oq[:80], "in_source": found,
            "jaccard": round(best_j, 3), "token_dev": round(1-best_j, 3),
            "cosine": round(best_cos, 3), "sem_dev": round(1-best_cos, 3),
            "gap": round((1-best_j) - (1-best_cos), 3)
        })
    return {"source_quotes": src_q, "output_quotes": out_q,
            "checks": results, "fabricated": sum(1 for r in results if not r["in_source"])}

print("Scorers defined.")

In [ ]:
print("Running Pipeline B (scored)...\n")
scored_output = run_pipeline(BROADCAST_SCRIPT, llm, "scored")

# Layer 1: Token-level quote check
scored_tokens = token_quote_check(BROADCAST_SCRIPT, scored_output)
# Layer 2: Three-axis LLM scorer
scored_axes = three_axis_scorer(BROADCAST_SCRIPT, scored_output)

print("PIPELINE B OUTPUT")
print("="*60)
print(scored_output)

print(f"\nTHREE-AXIS SCORES")
print(f"  Quote fidelity:       {scored_axes.get('quote_fidelity','?')}/5")
print(f"  Attribution accuracy: {scored_axes.get('attribution_accuracy','?')}/5")
print(f"  Semantic fidelity:    {scored_axes.get('semantic_fidelity','?')}/5")
print(f"  Decision:             {scored_axes.get('decision','?')}")
for d in scored_axes.get('deviations', []):
    print(f"    • {d}")

print(f"\nTOKEN-LEVEL QUOTE VERIFICATION")
print(f"  Fabricated: {scored_tokens['fabricated']}")
for c in scored_tokens['checks']:
    icon = '✓' if c['in_source'] else '✗'
    print(f"  [{icon}] \"{c['quote']}\"")
    print(f"      Jaccard={c['jaccard']} Cosine={c['cosine']} Gap={c['gap']}")

---
## Part 5 — MANDATORY HUMAN DECISION NODE

In [ ]:
# ============================================================
# MANDATORY HUMAN DECISION NODE — HARD HALT
# ============================================================
# The three-axis scorer assumes the LLM can reliably
# distinguish faithful reproduction from semantic drift.
#
# WHAT THE AI PROPOSED:
#   Bookie: "Use RAGAS context adherence scoring —
#   embedding cosine similarity between generated output
#   and retrieved context."
#
# WHAT I CORRECTED:
#   The 0.681 gap proves this is insufficient.
#   Cosine scores the fabricated Reyes quote as HIGH
#   adherence (same topic, same person, same vote).
#   Token overlap catches it (hedging words absent).
#
#   I built three axes instead:
#     Axis 1 (Quote Fidelity) → paraphrase-as-quote
#     Axis 2 (Attribution)    → source misattribution
#     Axis 3 (Semantic)       → framing shifts
#   Each axis maps to a different architectural fix.
#
# VALIDATION:
#   1. Check: did the scorer catch deviations Pipeline A missed?
#   2. Check: would generic RAGAS have caught them?
#   3. If false-proceed > 10%, add few-shot examples.
# ============================================================

print("⬡ HUMAN DECISION NODE — EXECUTION HALTED")
print("="*60)
print("Review the scored output above.")
print("1. Did the token checker catch any fabricated quotes?")
print("2. Did the three-axis scorer flag deviations?")
print("3. Would generic RAGAS cosine have caught them?")
print("4. Is the PASS/FLAG decision correct?")
print("")
print("AI CORRECTION: Bookie proposed generic RAGAS.")
print("I built three-axis scoring because the 0.681 gap proves")
print("generic metrics miss domain-specific failures.")
print("="*60)

# === HARD HALT — notebook stops here until human types a decision ===
human_decision = input("\nEnter PASS or FLAG after reviewing scores above: ")
print(f"\nHuman decision recorded: {human_decision}")
print("Proceeding to comparative analysis...")

---
## Part 6 — Retroactive scoring of Pipeline A (triggering the failure)

In [ ]:
# ============================================================
# Apply the REAL scorer to Pipeline A's UNSCORED output.
# This reveals deviations that were invisible without measurement.
# THIS IS THE TRIGGERABLE FAILURE.
# ============================================================

print("Retroactively scoring Pipeline A's unscored output...\n")

retro_tokens = token_quote_check(BROADCAST_SCRIPT, naive_output)
retro_axes = three_axis_scorer(BROADCAST_SCRIPT, naive_output)

print("RETROACTIVE THREE-AXIS SCORES ON PIPELINE A")
print("="*60)
print(f"  Quote fidelity:       {retro_axes.get('quote_fidelity','?')}/5")
print(f"  Attribution accuracy: {retro_axes.get('attribution_accuracy','?')}/5")
print(f"  Semantic fidelity:    {retro_axes.get('semantic_fidelity','?')}/5")
print(f"  Decision:             {retro_axes.get('decision','?')}")
for d in retro_axes.get('deviations', []):
    print(f"    • {d}")

print(f"\nRETROACTIVE QUOTE VERIFICATION (per-quote Jaccard vs Cosine)")
print("="*60)
print(f"  Fabricated quotes found: {retro_tokens['fabricated']}\n")
for c in retro_tokens['checks']:
    icon = '✓' if c['in_source'] else '✗ FABRICATED'
    print(f"  [{icon}] \"{c['quote']}\"")
    print(f"      Token dev: {c['token_dev']:.3f}  |  Sem dev: {c['sem_dev']:.3f}  |  Gap: {c['gap']:.3f}")
    if c['gap'] > 0.3:
        print(f"      ↑ Gap > 0.3 — domain-specific failure invisible to semantic metric")

print(f"\n{'='*60}")
print("These deviations EXISTED in Pipeline A's output.")
print("Without the scorer, they would have shipped to publication.")
print("The scorer works. The architecture just wasn't there.")

---
## Part 7 — Model-swap simulation (failure anatomy Stages 4-5)

The chapter's failure anatomy: after a published fabrication, the team replaces the model. Error rate drops from 3% to 2.1%. But the **measurement gap persists** because the architecture was never fixed.

Here we simulate this by running Pipeline A at two different temperatures (simulating a model swap). The measurement layer is still absent in both runs.

In [ ]:
# ============================================================
# MODEL SWAP SIMULATION
# Run Pipeline A with two different temperatures.
# Show that the measurement gap persists regardless of model.
# ============================================================

model_a = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0.5)  # "Old model"
model_b = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0.1)  # "New model"

print("Model A (temp=0.5) — simulates the 'old model'")
print("Model B (temp=0.1) — simulates the 'new, more capable model'")
print("Neither has a measurement layer.\n")

output_a = run_pipeline(BROADCAST_SCRIPT, model_a, "swap_a")
output_b = run_pipeline(BROADCAST_SCRIPT, model_b, "swap_b")

# Score both retroactively
tokens_a = token_quote_check(BROADCAST_SCRIPT, output_a)
tokens_b = token_quote_check(BROADCAST_SCRIPT, output_b)

print("MODEL A (temp=0.5) — retroactive quote check")
print(f"  Fabricated quotes: {tokens_a['fabricated']}")
for c in tokens_a['checks']:
    icon = '✓' if c['in_source'] else '✗'
    print(f"  [{icon}] \"{c['quote']}\" (token_dev={c['token_dev']}, sem_dev={c['sem_dev']})")

print(f"\nMODEL B (temp=0.1) — retroactive quote check")
print(f"  Fabricated quotes: {tokens_b['fabricated']}")
for c in tokens_b['checks']:
    icon = '✓' if c['in_source'] else '✗'
    print(f"  [{icon}] \"{c['quote']}\" (token_dev={c['token_dev']}, sem_dev={c['sem_dev']})")

print(f"\n{'='*60}")
print("FAILURE ANATOMY STAGES 4-5 DEMONSTRATED:")
print("The 'new model' may have fewer fabrications.")
print("But WITHOUT a measurement layer, NEITHER pipeline knows.")
print("The architectural deficiency persists across model swaps.")
print("The model was blamed. The architecture was the cause.")
print("="*60)

---
## Part 8 — Side-by-side comparison

In [ ]:
print("COMPARATIVE ANALYSIS")
print("="*65)
print(f"{'Criterion':<28} {'Pipeline A (Naive)':<22} {'Pipeline B (Scored)'}")
print("─"*70)
print(f"{'Model':<28} {'Claude Sonnet':<22} {'Claude Sonnet (same)'}")
print(f"{'Retrieval':<28} {'Top-5 cosine':<22} {'Top-5 cosine (same)'}")
print(f"{'Quote verification':<28} {'ABSENT':<22} {'Token + LLM'}")
print(f"{'Attribution check':<28} {'ABSENT':<22} {'Three-axis'}")

retro_dev_count = len(retro_axes.get('deviations', []))
scored_dev_count = len(scored_axes.get('deviations', []))

print(f"{'Deviations found':<28} {'0 (not measured)':<22} {scored_dev_count} scored")
print(f"{'Retro deviations (A)':<28} {retro_dev_count:<22} {'n/a (scored at gen)'}")
print(f"{'Decision':<28} {'None (absent)':<22} {scored_axes.get('decision','?')}")
print(f"\nSame model. Same script. Different architecture.")
print(f"Architecture is the leverage point, not the model.")

---
## Part 9 — Exercise: disable the scorer

In [ ]:
def scorer_DISABLED(source, generated):
    """BROKEN: always PASS. Simulates most production RAG systems."""
    return {"quote_fidelity": 5, "attribution_accuracy": 5, "semantic_fidelity": 5,
            "deviations": [], "decision": "PASS", "fix_suggestions": []}

print("EXERCISES:")
print("1. Run scorer_DISABLED on Pipeline A output. Compare to real scorer.")
print("2. Replace three_axis_scorer with pure cosine: does it catch misquotes?")
print("3. Design domain-specific metrics for legal/medical/financial RAG.")
print("\nTo run exercise 1:")
print("  disabled = scorer_DISABLED(BROADCAST_SCRIPT, naive_output)")
print("  real = three_axis_scorer(BROADCAST_SCRIPT, naive_output)")
print("  # Compare disabled['deviations'] vs real['deviations']")

---
## Part 10 — AI scaffold: evaluation architecture proposer

In [ ]:
def propose_evaluation(domain_desc):
    """AI scaffold: proposes domain-specific evaluation. Halts for review."""
    prompt = f"""Propose 5 domain-specific evaluation dimensions for a RAG agent.
For each: name, failure mode detected, scoring method, threshold, and why
generic RAGAS would miss it. JSON format.

Domain: {domain_desc}"""
    resp = llm.invoke([HumanMessage(content=prompt)])
    print("AI SCAFFOLD — PROPOSED EVALUATION")
    print("="*60)
    print(resp.content)
    print(f"\n{'='*60}")
    print("⬡ HUMAN DECISION NODE")
    print("Are these the failures that MATTER, or just easy to measure?")
    decision = input("Enter APPROVE, MODIFY, or REJECT: ")
    return {"proposal": resp.content, "human_decision": decision}

# Uncomment to run:
# propose_evaluation("Medical discharge summary generator")

---
## Summary

1. **0.681 gap** — token overlap catches what semantic similarity misses (computed live)
2. **Same model, different architecture** → different trust guarantees
3. **Retroactive scoring** proves Pipeline A had invisible deviations
4. **Model swap** doesn't fix the measurement gap (Stages 4-5 of failure anatomy)
5. **Per-quote Jaccard vs. cosine** shows the gap on live generated output

**Architecture is the leverage point. The model just executes what you designed.**